# Resultados da ablação

Análise pós-treino (rode `2_run_ablation.py` antes).

| Seção | O que compara |
|-------|----------------|
| **0. Auditoria** | Contagem wide→final e remoções por estágio (raw / mrmr / mrmr_stable) |
| **1. Eficiência** | AUC pooled entre tarefas, modelos e modalidades |
| **2. Consistência** | Estabilidade temporal dos biomarcadores (T1/T2/T3 agrupados) |
| **3. Filtro temporal** | Pós-hoc: biomarcadores com ≥2 visitas @ ≥70% de incidência |

**Entrada:** `csvs/longitudinal_4_groups/ablation_results/{modality}/ablation_results_all.csv`

**Saída:** figuras em `csvs/longitudinal_4_groups/ablation_results/figures/{selection}/`


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from ablation_analysis import (
    anatomical_key,
    count_stable_timepoints,
    feature_freq_table,
    feature_freq_table_grouped,
    filter_ablation_config,
    filter_temporally_stable,
    plot_compare_stability,
    plot_temporal_stability_lines,
    prepare_ablation_df,
    selection_audit_report,
    summarize_selection_audit,
)

# --- config ---
RESULTS_ROOT = Path("csvs/longitudinal_4_groups/ablation_results")
SELECTION_MODE = "mrmr_stable"  # raw | filters | mrmr — deve existir nos CSVs
WITH_COMBAT = False  # True | False | None (ambos)

# modalidades com pasta de resultados; None = auto-detect
MODALITIES = None  # ex.: ("vol", "shape")

MODEL_ORDER = ["svm", "rf", "mlp"]
MOD_ORDER = ["vol", "shape", "texture", "disp", "all"]
TASK_ORDER = ["cn_ad", "cn_smci", "smci_pmci", "smci_ad", "cn_pmci", "pmci_ad"]
TASK_LABELS = {
    "cn_ad": "CN × AD",
    "cn_smci": "CN × sMCI",
    "smci_pmci": "sMCI × pMCI",
    "smci_ad": "sMCI × AD",
    "cn_pmci": "CN × pMCI",
    "pmci_ad": "pMCI × AD",
}
COLOR_MODEL = {"svm": "#4477AA", "rf": "#EE6677", "mlp": "#228833"}
COLOR_MOD = {"vol": "#4477AA", "shape": "#EE6677", "texture": "#228833", "disp": "#CCBB44", "all": "#AA3377"}

SAVE_FIGS = True
FIG_DPI = 150
FIGURES_DIR = RESULTS_ROOT / "figures" / SELECTION_MODE

# estabilidade (seção 2)
STAB_TASK = "smci_pmci"
STAB_MODEL = "rf"
STAB_MODALITY = "vol"  # uma modalidade por gráfico
STAB_COMBAT = False
MIN_COVERAGE_PCT = 70
STAB_ALL_CONFIGS = False  # True = gera estabilidade para task×modelo×modalidade
EXPECTED_FOLDS = 5

COLS_SHOW = [
    "feature_short", "coverage_pct", "pct_T1", "pct_T2", "pct_T3",
    "amplitude", "n_folds_any", "total_selections",
]

# auditoria (seção 0)
AUDIT_TASK = STAB_TASK
AUDIT_MODEL = STAB_MODEL
AUDIT_MODALITY = STAB_MODALITY
AUDIT_COMBAT = STAB_COMBAT
AUDIT_SELECTION_MODES = None  # None = todos os modos presentes no CSV

# filtro temporal pós-hoc (seção 3)
TEMPORAL_MIN_PCT = 70
TEMPORAL_MIN_TIMEPOINTS = 2


def resolve_csv(mod_dir: Path, stem: str) -> Path:
    for name in (f"{stem}_{SELECTION_MODE}.csv", f"{stem}.csv"):
        path = mod_dir / name
        if path.is_file():
            return path
    raise FileNotFoundError(f"{stem} ausente em {mod_dir} (selection={SELECTION_MODE})")


def available_modalities(root: Path) -> list[str]:
    mods = []
    for p in sorted(root.iterdir()):
        if not p.is_dir() or p.name in ("figures",):
            continue
        try:
            resolve_csv(p, "ablation_summary")
            mods.append(p.name)
        except FileNotFoundError:
            pass
    return mods


def load_summaries(modalities: list[str]) -> pd.DataFrame:
    frames = [pd.read_csv(resolve_csv(RESULTS_ROOT / m, "ablation_summary")) for m in modalities]
    return pd.concat(frames, ignore_index=True)


def load_ablation(modalities: list[str]) -> pd.DataFrame:
    frames = [pd.read_csv(resolve_csv(RESULTS_ROOT / m, "ablation_results_all")) for m in modalities]
    return prepare_ablation_df(pd.concat(frames, ignore_index=True))


def prep_summary(summary: pd.DataFrame) -> pd.DataFrame:
    df = summary.copy()
    if "selection_mode" in df.columns:
        df = df[df["selection_mode"].astype(str) == SELECTION_MODE]
    if WITH_COMBAT is not None:
        df = df[df["with_combat"] == WITH_COMBAT]
    df["task_label"] = df["task"].map(TASK_LABELS).fillna(df["task"])
    df["model_key"] = pd.Categorical(df["model_key"].astype(str), categories=MODEL_ORDER, ordered=True)
    df["modality"] = pd.Categorical(df["modality"].astype(str), categories=MOD_ORDER, ordered=True)
    present_tasks = [t for t in TASK_ORDER if t in set(df["task"])]
    df["task"] = pd.Categorical(df["task"].astype(str), categories=present_tasks, ordered=True)
    return df.dropna(subset=["task", "model_key", "modality"])


def save_fig(fig, name: str) -> Path:
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    path = FIGURES_DIR / name
    if SAVE_FIGS:
        fig.savefig(path, dpi=FIG_DPI, bbox_inches="tight")
        print(f"Salvo: {path}")
    return path


LOADED_MODALITIES = MODALITIES or available_modalities(RESULTS_ROOT)
if not LOADED_MODALITIES:
    raise FileNotFoundError(f"Nenhuma modalidade com summary em {RESULTS_ROOT}")

summary_raw = load_summaries(LOADED_MODALITIES)
df_ablation = load_ablation(LOADED_MODALITIES)
plot_df = prep_summary(summary_raw)

combat_tag = {True: "combat", False: "nocombat", None: "all"}[WITH_COMBAT]
print(f"Modalidades: {LOADED_MODALITIES}")
print(f"Seleção: {SELECTION_MODE} | ComBat: {WITH_COMBAT}")
print(f"Configs: {len(plot_df)} | Figuras: {FIGURES_DIR}")
display(plot_df[["task", "modality", "model_key", "with_combat", "auc_mean", "auc_std", "auc_pooled"]].head(12))

## 0. Auditoria de seleção

Compara `raw`, `mrmr` e `mrmr_stable` na mesma config (task × modelo × ComBat).

In [ ]:
audit_modes = AUDIT_SELECTION_MODES
if audit_modes is None:
    m = (
        (df_ablation["task"].astype(str) == AUDIT_TASK)
        & (df_ablation["model_key"].astype(str) == AUDIT_MODEL)
        & (df_ablation["with_combat"] == AUDIT_COMBAT)
        & (df_ablation["modality"].astype(str) == AUDIT_MODALITY)
    )
    audit_modes = tuple(sorted(df_ablation.loc[m, "selection_mode"].astype(str).unique()))

audit_summary = selection_audit_report(
    df_ablation,
    task=AUDIT_TASK,
    model=AUDIT_MODEL,
    with_combat=AUDIT_COMBAT,
    modality=AUDIT_MODALITY,
    selection_modes=audit_modes,
)
display(audit_summary)

audit_detail = summarize_selection_audit(
    df_ablation[
        (df_ablation["task"].astype(str) == AUDIT_TASK)
        & (df_ablation["model_key"].astype(str) == AUDIT_MODEL)
        & (df_ablation["with_combat"] == AUDIT_COMBAT)
        & (df_ablation["modality"].astype(str) == AUDIT_MODALITY)
        & (df_ablation["selection_mode"].astype(str).isin(audit_modes))
    ]
)
display(audit_detail.head(10))


## 1. Eficiência dos modelos (AUC pooled)

Três vistas: **tarefa** × **modelo** × **modalidade**.

In [ ]:
def _cat_levels(series: pd.Series, preferred: list | None = None) -> list:
    if hasattr(series, "cat"):
        levels = list(series.cat.categories)
    else:
        levels = sorted(series.dropna().unique())
    if preferred:
        return [v for v in preferred if v in set(levels)] + [v for v in levels if v not in set(preferred)]
    return levels


def plot_auc_faceted(data: pd.DataFrame, *, x: str, hue: str, col: str, title: str, fname: str):
    sub = data.dropna(subset=[x, hue, col, "auc_pooled"])
    if sub.empty:
        print(f"Sem dados: {fname}")
        return
    cols = _cat_levels(sub[col])
    ncols = max(len(cols), 1)
    fig, axes = plt.subplots(1, ncols, figsize=(4.2 * ncols, 4), squeeze=False)
    palette = COLOR_MODEL if hue == "model_key" else (COLOR_MOD if hue == "modality" else None)
    for ax, cat in zip(axes.flat, cols):
        part = sub[sub[col].astype(str) == str(cat)]
        order = _cat_levels(
            part[x],
            MOD_ORDER if x == "modality"
            else TASK_ORDER if x == "task"
            else [TASK_LABELS[t] for t in TASK_ORDER] if x == "task_label"
            else None,
        )
        hue_pref = MODEL_ORDER if hue == "model_key" else MOD_ORDER if hue == "modality" else None
        hue_order = _cat_levels(part[hue], hue_pref)
        sns.barplot(
            data=part, x=x, y="auc_pooled", hue=hue,
            order=order, hue_order=hue_order, palette=palette,
            ax=ax, errorbar=None,
        )
        ax.set_title(str(cat), fontsize=10)
        ax.set_ylabel("AUC pooled")
        ax.set_ylim(0.4, 1.0)
        ax.axhline(0.5, color="gray", ls="--", lw=0.8)
        ax.tick_params(axis="x", rotation=25, labelsize=8)
        if ax.get_legend():
            ax.legend_.remove()
    handles, labels = axes.flat[0].get_legend_handles_labels()
    if not handles:
        handles, labels = axes.flat[-1].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc="upper center", ncol=min(len(labels), 4), bbox_to_anchor=(0.5, 1.02))
    fig.suptitle(title, fontsize=12, y=1.06)
    plt.tight_layout()
    save_fig(fig, fname)
    plt.show()
    plt.close(fig)


# 1a) Por tarefa binária: em cada task, comparar modelos e modalidades
plot_auc_faceted(
    plot_df, x="modality", hue="model_key", col="task_label",
    title=f"AUC pooled por modalidade e modelo — por tarefa ({combat_tag}, {SELECTION_MODE})",
    fname=f"auc_by_task_modality_model_{combat_tag}.png",
)

# 1b) Por modelo: em cada modelo, comparar tarefas e modalidades
plot_auc_faceted(
    plot_df, x="modality", hue="task_label", col="model_key",
    title=f"AUC pooled por modalidade e tarefa — por modelo ({combat_tag}, {SELECTION_MODE})",
    fname=f"auc_by_model_modality_task_{combat_tag}.png",
)

# 1c) Por modalidade: em cada modalidade, comparar tarefas e modelos
plot_auc_faceted(
    plot_df, x="task_label", hue="model_key", col="modality",
    title=f"AUC pooled por tarefa e modelo — por modalidade ({combat_tag}, {SELECTION_MODE})",
    fname=f"auc_by_modality_task_model_{combat_tag}.png",
)

# 1d) Heatmap overview: modelo × modalidade, painel por tarefa
tasks = [t for t in TASK_ORDER if t in set(plot_df["task"].astype(str))]
fig, axes = plt.subplots(1, len(tasks), figsize=(4 * len(tasks), 3.5), squeeze=False)
for ax, task_id in zip(axes.flat, tasks):
    sub = plot_df[plot_df["task"].astype(str) == task_id]
    pivot = sub.pivot_table(
        index="model_key", columns="modality", values="auc_pooled", aggfunc="first", observed=True
    )
    pivot = pivot.reindex(index=MODEL_ORDER, columns=[m for m in MOD_ORDER if m in pivot.columns])
    sns.heatmap(pivot, annot=True, fmt=".2f", cmap="YlOrRd", vmin=0.5, vmax=1.0, ax=ax, cbar=len(tasks) == 1)
    ax.set_title(TASK_LABELS.get(task_id, task_id), fontsize=9)
fig.suptitle(f"AUC pooled — heatmap modelo × modalidade ({combat_tag}, {SELECTION_MODE})", fontsize=11)
plt.tight_layout()
save_fig(fig, f"auc_heatmap_task_model_modality_{combat_tag}.png")
plt.show()
plt.close(fig)

# Tabela ranqueada
rank = plot_df.sort_values("auc_pooled", ascending=False)[
    ["task_label", "modality", "model_key", "auc_mean", "auc_std", "auc_pooled", "n_features_mean"]
]
print("\nTop configs (AUC pooled):")
display(rank.head(15))

## 2. Consistência dos atributos

Tabelas: todos vs **estáveis** (`filter_temporally_stable`).

**Plots** (`temporal_stable_*`, `compare_stable_*`) usam só biomarcadores em `grp_stable`; por fold, features abaixo do threshold saem de `selected_features`.

Ajuste `STAB_*`, `TEMPORAL_*` ou `STAB_ALL_CONFIGS = True`.


In [ ]:
def _cfg_only_stable(df_cfg: pd.DataFrame, grp_stable: pd.DataFrame) -> pd.DataFrame:
    """Mantém só features cujo anatomical_key passou no filtro temporal."""
    if grp_stable.empty:
        return df_cfg.iloc[0:0].copy()
    allowed = set(grp_stable["feature_group"].astype(str))
    rows = []
    for _, row in df_cfg.iterrows():
        kept = [f for f in json.loads(row["selected_features"]) if anatomical_key(f) in allowed]
        if not kept:
            continue
        d = row.to_dict()
        d["selected_features"] = json.dumps(kept)
        rows.append(d)
    return pd.DataFrame(rows)


def run_stability(task_id: str, mod_id: str, model_key: str, with_combat: bool) -> None:
    sub_all = df_ablation[df_ablation["modality"].astype(str) == mod_id]
    if sub_all.empty:
        print(f"Pulando {mod_id}: sem dados carregados")
        return
    try:
        df_cfg = filter_ablation_config(
            sub_all,
            task=task_id,
            modality=mod_id,
            model_key=model_key,
            with_combat=with_combat,
            selection_mode=SELECTION_MODE,
            expected_folds=EXPECTED_FOLDS,
        )
    except ValueError as exc:
        print(f"Pulando {task_id}/{mod_id}/{model_key}/combat={with_combat}: {exc}")
        return

    tag = f"{task_id}_{mod_id}_{model_key}_{'combat' if with_combat else 'nocombat'}_{SELECTION_MODE}"
    stab_dir = FIGURES_DIR / "stability"
    stab_dir.mkdir(parents=True, exist_ok=True)
    print(f"\n=== {tag} | {len(df_cfg)} avaliações ===")

    grp = feature_freq_table_grouped(df_cfg)
    grp["_n_tp_70"] = count_stable_timepoints(grp, min_pct=TEMPORAL_MIN_PCT)

    print(f"\nTodos selecionados (top 20 de {len(grp)}):")
    display(grp[COLS_SHOW + ["_n_tp_70"]].head(20))

    grp_stable = filter_temporally_stable(
        grp,
        min_pct=TEMPORAL_MIN_PCT,
        min_timepoints=TEMPORAL_MIN_TIMEPOINTS,
    )
    print(
        f"\nEstáveis temporalmente (>= {TEMPORAL_MIN_TIMEPOINTS} tp @ >={TEMPORAL_MIN_PCT}%): "
        f"{len(grp_stable)} / {len(grp)}"
    )
    if grp_stable.empty:
        print("(nenhum — plots omitidos)")
        if SAVE_FIGS:
            grp.to_csv(stab_dir / f"feature_freq_grouped_{tag}.csv", index=False)
            grp_stable.to_csv(stab_dir / f"feature_freq_stable_{tag}.csv", index=False)
        return
    display(grp_stable[COLS_SHOW])

    df_stable = _cfg_only_stable(df_cfg, grp_stable)
    plot_suffix = (
        f"estáveis (>={TEMPORAL_MIN_TIMEPOINTS} tp @ >={TEMPORAL_MIN_PCT}%) | "
        f"{len(grp_stable)} biomarcadores"
    )

    lines_path = stab_dir / f"temporal_stable_{tag}.png"
    fig = plot_temporal_stability_lines(
        df_stable,
        freq=grp_stable,
        title=f"{tag} | {plot_suffix}",
        out_path=lines_path if SAVE_FIGS else None,
        sort_by="feature_short",
    )
    plt.show()
    plt.close(fig)
    if SAVE_FIGS:
        print(f"Salvo: {lines_path}")

    compare_path = stab_dir / f"compare_stable_{tag}.png"
    fig = plot_compare_stability(
        df_stable,
        title=f"{tag} | {plot_suffix}",
        out_path=compare_path if SAVE_FIGS else None,
    )
    plt.show()
    plt.close(fig)

    # if SAVE_FIGS:
    #     grp.to_csv(stab_dir / f"feature_freq_grouped_{tag}.csv", index=False)
    #     grp_stable.to_csv(stab_dir / f"feature_freq_stable_{tag}.csv", index=False)
    #     feature_freq_table(df_stable).to_csv(stab_dir / f"feature_freq_stable_{tag}.csv", index=False)
    #     print(f"Salvo: {compare_path}")


if STAB_ALL_CONFIGS:
    combats = [False, True] if WITH_COMBAT is None else [WITH_COMBAT]
    for task_id in sorted(df_ablation["task"].astype(str).unique()):
        for mod_id in LOADED_MODALITIES:
            for model_key in MODEL_ORDER:
                for wc in combats:
                    run_stability(task_id, mod_id, model_key, wc)
else:
    run_stability(STAB_TASK, STAB_MODALITY, STAB_MODEL, STAB_COMBAT)
